In [2]:
import numpy as np
from sklearn.linear_model import SGDClassifier

# Homework 5

In this final homework, we practice what we have learned about in PAC learning, linear classification, and linear regression.

## Problem 1

Recall that we have discussed PAC learning intervals in class, where the concept class $\mathcal{C}$ contains indicators of closed intervals $[a,b] \subseteq [0,1]$. The sample complexity to achieve an $(\epsilon, \delta)$ guarantee for an interval is $n = \lceil \frac{2}{\epsilon} \log(\frac{2}{\delta}) \rceil$. 

Consider now a $2$-dimensional space $\mathcal{X} = [0,1]^2$. Let the concept class $\mathcal{C}$ be the set of all axis-aligned rectangles in this space. A target concept $c_* \in \mathcal{C}$ is defined by $2$ intervals, one for each dimension: $[a_*^1, b_*^1] \times [a_*^2, b_*^2]$. A data point $\mathbf{x} = (x_1, x_2)$ is labeled positive ('$1$') if $x_i \in [a_*^i, b_*^i]$ for all $i \in \{1, 2\}$, and negative ('$0$') otherwise.

1. Propose an algorithm to learn this concept class from $n$ i.i.d. examples drawn from an unknown distribution $\mathcal{D}$.
2. Derive the sample complexity $n$ required for your algorithm to guarantee that with probability at least $1 - \delta$, the approximation error is at most $\epsilon$. Show your work.

First let $L = [a_*^1, b_*^1]\times [a_*^2, b_*^2]$ and let $sl^y(L) = \{x: (x,y) \in L\}$ and $sl_x(L) = \{y, (x,y) \in L\}$ denote the generalized axis aligned rectangles of $L$. Then define
> i) $a^1 = min(x \in sl^y(L))$\
ii)$b^1 = max(x \in sl^y(L))$\
ii) $a^2 = min(x \in sl_x(L))$\
iv) $b^2 = max(x \in sl_x(L)).$

Observe that $\mathbf{1}_{\hat{B}} = \mathbf{1}_{[a^1,b^2]} \times \mathbf{1}_{[a^2, b^2]}$, thus $err(\mathbf{1}_{\hat{B}}) = \int_{\mathcal{X}} \mathbf{1}_{L} - \mathbf{1}_{\hat{B}} = m(L\backslash \hat{B})$ for probability measure m. Now let $E_S = \bigcup_{i=1}^n (x_i \cap (sl^y(\mathcal{X} \backslash L) \times sl_x(\mathcal{X} \backslash L)))$. Then since $L$ is bounded on 4 sides then by subadditivity, 
$$m(L\backslash \hat{B}) \leq m(\cup_{S=1}^4 E_S) \leq \sum_{S=1}^4 m(E_S) = \epsilon$$ for $m(E_S) \leq \frac{\epsilon}{4}$. In other words, union bound tells us 
$$\mathbb{P}[L \backslash \hat{B} \neq \{\emptyset\}] \leq 4(1-\frac{\epsilon}{4})^n \leq \delta$$
for $n = \frac{4}{\epsilon}\log(\frac{4}{\delta})$ samples.

 

## Problem 2

In Lecture note 7, we talked about the Perceptron algorithm. Consider the same setting as in that lecture note, but now assume that you are given a promise that all but an $\epsilon \in (0, 1/2)$ fraction of the labeled examples satisfy the $\gamma$-margin condition for some $\gamma > 0.$ You are given the value of $\epsilon,$ but not the value of $\gamma.$ Consider the following modification of the Perceptron algorithm: in each iteration $t$, let $\mathcal{S}_t$ denote the set of indices of misclassified examples for the current iterate $\mathbf{w}_t.$ If $|\mathcal{S}_t|>2\epsilon n,$ update $\mathbf{w}_{t+1} = \mathbf{w}_t + \frac{1}{|\mathcal{S}_t|}\sum_{i \in \mathcal{S}_t}y_i \mathbf{x}_i;$ otherwise the algorithm halts.

1. Write the code that implements this algorithm using a while loop (without an "if statement" that halts the algorithm), similar to the original statement of Perceptron, but accounting for the above modification. The algorithm should take the sample set $(x_i, y_i),$ $i \in \{1, \dots, n\}$ and $\epsilon \in (0, 1/2)$ as its input. It should output the last computed $\mathbf{w}_t$.

2. Argue that the modified Perceptron  algorithm runs in at most $\lceil \frac{4}{\gamma^2}\rceil$ iterations, after which it outputs a linear separator that misclassifies at most a $2\epsilon$ fraction of the input labeled examples. 

In [1]:
def perceptron(X, y, epsilon):
    
    n, d = X.shape
    w = np.zeros(d)
    
    margins = y*(X.dot(w))
    S = np.where(margins<=0)[0]
    
    """"Your while loop here"""

    return w


## Problem 3

Consider the _realizable_ setting of linear regression, where you are given a promise that there exists a vector $\mathbf{w}_* \in \mathbb{R}^d$ with $\|\mathbf{w}_*\|_2 \leq 1$ such that $y = \mathbf{w}_*^\top \mathbf{x}$ for any sample $(\mathbf{x}, y)$ that can possibly be drawn from an unknown distribution $\mathcal{D}$. You can assume that for each such sample you have $\|\mathbf{x}\|_2 \leq 1$ and $|y| \leq 1.$ Argue that the ordinary least squares estimator $\mathbf{\hat{w}}$ using $n = O\big(\frac{d\log(d/\epsilon) + \log(1/\delta)}{\epsilon^2}\big)$ labeled examples drawn i.i.d. from $\mathcal{D}$, where $\epsilon, \delta \in (0,1),$ can guarantee that with probability at least $1 - \delta,$

$$
    \mathbb{E}_{(\mathbf{x}, y)  \sim \mathcal{D}}[(y - \mathbf{\hat{w}}^\top \mathbf{x})^2] \leq \epsilon.
$$
Note here that unlike the lecture notes, you should not be assuming that $\mathbf{\hat{w}}$ is necessarily quantized.

Notice that the hypothesis class $\mathcal{H}$ is a unit ball so there exists a set $N = \{\bar{\mathbf{w}}; \|\mathbf{w} - \mathbf{\bar{w}}\|_2 \leq \alpha\}$. Since $vol(N) \leq 2d(1+\frac{2}{\alpha})^{d-1}$ then we know $|N| \leq \frac{3}{\alpha}$. It suffices to show that 
$$\mathbb{P}[max |err(\mathbf{w}_* - err(\mathbf{\bar{w}})| \geq \frac{\epsilon}{4}] \leq (\frac{3}{\alpha})^d 2e^\frac{n\epsilon^2}{8} \leq \delta$$.
First observe with some algebra and application of cauchy schwarz for expectation that 
$$|err(\mathbf{w}_*) - err(\bar{\mathbf{w}})| \leq \mathbb{E}[|2y-(\mathbf{w}_* + \mathbf{\bar{w}})^T \mathbf{x}| |(\mathbf{w}_* - \mathbf{\bar{w}})^T \mathbf{x}|] \leq 4\alpha.$$ Then for $\alpha = \frac{\epsilon}{32}$, $n = O\big(\frac{d\log(d/\epsilon) + \log(1/\delta)}{\epsilon^2}\big)$ many samples follows as the minimum number of samples ensuring convergence in the order of $\frac{\epsilon}{8}$.

## Coding Assignment

## Problem 4: Predicting Computing Job Failures at CHTC

The Center for High Throughput Computing (CHTC) at UW-Madison handles thousands of distributed computing jobs every day for researchers. Suppose you work as a systems data scientist for the CHTC and are tasked with developing a new diagnostic tool. Your goal is to predict whether a newly submitted machine learning job will successfully complete or if it will fail (e.g., due to Out-Of-Memory errors or timeout constraints) based on the job's configuration. You'll try to do this based on "yes" or "no" type information on the many settings or knobs you can configure for each job on CHTC.

In particular, given a job's profile (requested RAM or not, requested CPU or not, script size big or small, dataset big or small, etc.) encoded in a vector $x$ in $\{-1, +1\}^d$ where $d = 100$ (that is, each job profile is a 100-dimensional vector with each row in the vector being either -1 or +1), you want to develop a classifier that outputs −1 when the job will fail and +1 when the job will succeed.


### Training Phase


You have $n$ historical job profiles in a CSV file named `training-features.csv`. After reviewing the server logs, you also know whether these jobs succeeded or failed, and the labels are saved in `training-labels.csv`.

We provide an example code for training linear classifers ([link to documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html#sklearn.linear_model.SGDClassifier)).

```
np.random.seed(0)  # Fixing a random seed

from sklearn.linear_model import SGDClassifier
X = [[-1, +1], [+1, -1], [+1, +1], [-1, -1]]  # Training features
Y = [-1, 1, 1, -1]  # Training labels

# Define the classifer to train
clf = SGDClassifier(max_iter=1000, tol=1e-4)

# Train the classifer
clf.fit(X, Y)
```

1. Write a function that returns a trained linear classifer object using the training data provided. You should use `SGDClassifier` from `sklearn.linear_model` and train it with max number of iterations `max_ter = 1000` and tolerance `tol = 1e-4`. You should also initiate random seed to $0$ by calling `np.random.seed(0)` before training your classifer.

In [4]:
# Run this cell to read training data into notebook.
# Please put the csv files in the same directory as this notebook.

training_features = np.genfromtxt("./training-features.csv", delimiter=",")
training_labels = np.genfromtxt("./training-labels.csv", delimiter=",")

In [5]:
def train_classifier(training_features, training_labels):
    """
    Parameters
    ----------
    training_features: numpy.array
        Features of the training dataset. Shape = (n, d)
    training_labels: numpy.array
        Labels of the training dataset. Shape = (n,)
        
    Returns
    -------
    SGDClassifier
        A trained classifier.
    """


    

2. How large does $n$ need to be so that with confidence 95% you can guarantee that
$$\mathbb{P}_{(x, y)\sim \mathcal{D}} [h(x) \neq y] \le \epsilon$$ for $\epsilon = 1e-3$, where $h$ is a linear classifer (a linear treshold function as in Lecture 4)? Here and in the rest of the question, you can take $c = 2$ to get numerical values that are being asked. 

3. Now suppose that you cannot gather as much historical log data as required in part 2, but you have n=8000 job logs (the actual number of examples in the training dataset). How small can you make $\epsilon$?

4. Given that $d = 100,$ discuss whether it would be possible to train a linear classifier with zero error (in terms of the population loss) in your lifetime.

### Testing Phase


After developing the diagnostic test, the CHTC evaluates how well the test performs in practice by deploying it to a trial server queue. As new jobs are submitted, you run the test to make a prediction of whether the job will succeed or fail.

In this trial phase, we have N new jobs and their features are recorded in `testing-features.csv`. After the jobs actually run, the true success/fail results are recorded in `testing-labels.csv`.

5. Write a function that makes predictions for the jobs in the trial phase using the linear classifier you trained, then compute the empirical misclassification error.

Hint: You can call `clf.predict([x])` to make a prediction for input feature `[x]` using your linear classifier `clf`.

In [ ]:
# Run this cell to load the classifier into variable diagnostic and read testing data into notebook.
# Please put the csv files in the same directory as this notebook.

diagnostic = train_classifier(training_features, training_labels)  # Your linear classifier

testing_features = np.genfromtxt("./testing-features.csv", delimiter=",")
testing_labels = np.genfromtxt("./testing-labels.csv", delimiter=",")
testing_features.shape

In [9]:
def compute_performance_quantities(diagnostic, testing_features, testing_labels):
    """
    Parameters
    ----------
    diagnostic: SGDClassifier
        A trained SGDClassifier linear classifier.
    testing_features: numpy.array
        Features of the testing dataset. Shape = (N, d)
    testing_labels: numpy.array
        Labels of the testing dataset. Shape = (N,)
    
    Returns
    -------
    (float, float, float)
        A tuple of specificity, sensitivity, miss.
    """
    

6. Compute the misclassification error using the above function you wrote for our diagnostic test. How does this compare to $\epsilon$ you computed in Part 3? Output $\epsilon_{\mathrm{miss}} / \epsilon$, where $\epsilon_{\mathrm{miss}}$ is the empirical misclassification error computed by your code, and discuss the result.